# 05 - Hybrid Ensemble Training
Notebook wrapper for src/train_hybrid.py.

Combines tabular features with GraphSAGE embeddings and trains final XGBoost ensemble.

In [ ]:
from pathlib import Path
from time import perf_counter
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Using project root: {PROJECT_ROOT}')

In [ ]:
from falabella_risk.models.train_hybrid import train_hybrid

train_hybrid(
    feature_path=Path('data/processed/features.parquet'),
    embedding_path=Path('data/processed/gnn_embeddings.parquet'),
    model_out=Path('models/hybrid_ensemble.pkl'),
    seed=42,
    mlflow_uri=None,
    experiment_name='falabella_hybrid_notebook',
    drop_leaky_features=True,
)

In [ ]:
import mlflow

exp = mlflow.get_experiment_by_name('falabella_hybrid_notebook')
if exp is None:
    print('Experiment not found yet.')
else:
    runs = mlflow.search_runs([exp.experiment_id], order_by=['start_time DESC'])
    display_cols = [
        'run_id',
        'metrics.auc', 'metrics.pr_auc', 'metrics.f1', 'metrics.brier', 'metrics.latency_ms_per_row',
        'metrics.calibrated_auc', 'metrics.calibrated_pr_auc', 'metrics.calibrated_f1',
        'metrics.calibrated_brier', 'metrics.calibrated_latency_ms_per_row',
        'params.calibration_threshold', 'metrics.val_best_f1',
    ]
    existing = [c for c in display_cols if c in runs.columns]
    runs[existing].head(1)

In [ ]:
import json
import joblib
import pandas as pd

model_path = Path('models/hybrid_ensemble.pkl')
threshold_path = Path('models/hybrid_ensemble.threshold.json')
print('Hybrid model exists:', model_path.exists())
print('Hybrid threshold metadata exists:', threshold_path.exists())

if threshold_path.exists():
    meta = json.loads(threshold_path.read_text())
    print('Calibrated threshold:', round(float(meta.get('decision_threshold', 0.5)), 4))

if model_path.exists():
    model = joblib.load(model_path)
    feat = pd.read_parquet('data/processed/features.parquet')
    emb = pd.read_parquet('data/processed/gnn_embeddings.parquet')
    df = feat.merge(emb, on='borrower_id', how='inner')
    x_all = df.drop(columns=['borrower_id', 'default_flag'], errors='ignore')

    if hasattr(model, 'feature_names_in_') and model.feature_names_in_ is not None:
        for col in model.feature_names_in_:
            if col not in x_all.columns:
                x_all[col] = 0.0
        x_all = x_all[list(model.feature_names_in_)]

    x = x_all.head(1000)

    t0 = perf_counter()
    _ = model.predict_proba(x)
    ms_per_row = (perf_counter() - t0) * 1000 / max(len(x), 1)
    print('Approx latency ms/row:', round(ms_per_row, 4))